### TP 3 - Analyse Exploratoire de Données (EDA) avec Python

In [ ]:
# Imports des libreries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from IPython.display import display, HTML

In [ ]:
# Verification est Imports
print("libreries importées avec succès.\n")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"seaborn : {sns.__version__}")

In [ ]:
# Configuration Style global
warnings.filterwarnings("ignore")
%matplotlib inline
plt.style.use('ggplot')

### Partie 1 : Chargement et Exploration Initiale

In [ ]:
# 1.1 Chargement du fichier CSV
DATA_PATH = os.path.join("..", "data", "raw", "sales_data.csv")
df = pd.read_csv(DATA_PATH)
print(f"dataset chager avec succes : {DATA_PATH}")
print(f"Dimensions                 : {df.shape[0]} lignes x {df.shape[1]} colonnes")

In [ ]:
# 1.2.1 Affichage des 5 premières lignes
print("5 premières lignes :")
display(HTML(df.head().to_html(index=False)))

In [ ]:
# 1.2.2 Types de données
print("Types de données :")
df_types = df.dtypes.reset_index()
df_types.columns = ['Donneés', 'Type']

display(HTML(df_types.to_html(index=False)))

In [ ]:
# Types de variables
numeriques = df.select_dtypes(include=[np.number]).columns.tolist()
categorielles = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print(f"Variables NUMÉRIQUES   : {numeriques}")
print(f"Variables CATÉGORIELLES: {categorielles}")

In [ ]:
# 1.2.3 Statistiques descriptives
print("Statistiques descriptives :")
df.describe().round(2)

In [ ]:
# 1.3.1 Valeurs manquantes
df_missing = pd.DataFrame({
    'Colonne': df.columns,
    'Valeurs_manquantes': df.isnull().sum().values,
})

print("valeurs manquantes:")
display(HTML(df_missing.to_html(index=False)))

In [ ]:
# 1.3.2 Doublons
print("Doublons :", df.duplicated().sum())

In [ ]:
# 1.3.3 Détection des incohérences
incoh_qte = df[df["Quantité"] <= 0]
incoh_prix = df[df["Prix_Unitaire"] <= 0]

print(f"Quantité ≤ 0  : {len(incoh_qte)} ligne(s)")
print(f"Prix_Unitaire ≤ 0 : {len(incoh_prix)} ligne(s)")

# Suppression si existants
df = df[(df["Quantité"] > 0) & (df["Prix_Unitaire"] > 0)]
df = df.reset_index(drop=True)


### Partie 2 : Nettoyage des Données

In [ ]:
# 2.1 Traitement des valeurs manquantes

#df = df.dropna()
#df.fillna("")

In [ ]:
# 2.2 suppretion des doublons

#df = df.drop_duplicates()

In [ ]:
print("Après nettoyage :", df.shape)

In [ ]:
# 2.3 Création du chiffre d'affaires
df["Chiffre_Affaires"] = df["Quantité"] * df["Prix_Unitaire"]
df.head()

### Partie 3 : Analyse Descriptive

In [ ]:
# 3.1.1 Chiffre d’affaires total
ca_total = df["Chiffre_Affaires"].sum()

# 3.1.2 Vente moyenne par transaction
vente_moyenne = df["Chiffre_Affaires"].mean()

# 3.1.3 Produit le plus vendu
produit_top = df.groupby("Produit")["Quantité"].sum().idxmax()

# 3.1.4 Catégorie la plus rentable
categorie_top = df.groupby("Catégorie")["Chiffre_Affaires"].sum().idxmax()

df_recap = pd.DataFrame({
    'Indicateur': ['Chiffre d\'affaires total', 'Vente moyenne par transaction', 
                   'Produit le plus vendu', 'Catégorie la plus rentable'],
    'Valeur': [f"{ca_total:.2f} €", f"{vente_moyenne:.2f} €",produit_top, categorie_top]
})

display(HTML(df_recap.to_html(index=False)))

In [ ]:
# 3.2.1 Répartition des ventes par ville

ventes_ville = df.groupby("Ville")["Chiffre_Affaires"].sum().sort_values(ascending=False)

df_ventes_ville = df.groupby("Ville").agg({
    'Chiffre_Affaires': ['sum', 'mean', 'count']
}).round(2)

df_ventes_ville.columns = ['CA_Total', 'CA_Moyen', 'Nb_Transactions']
df_ventes_ville = df_ventes_ville.sort_values('CA_Total', ascending=False).reset_index()

print("\nVentes par ville:")
display(HTML(df_ventes_ville.to_html(index=False)))

In [ ]:
# 3.2.1 Évolution des ventes dans le temps (mensuelle)
df["Date"] = pd.to_datetime(df["Date"])
df["Mois"] = df["Date"].dt.to_period("M")
ventes_mensuelles = df.groupby("Mois")["Chiffre_Affaires"].sum()

df_ventes_mensuelles = pd.DataFrame({
    'Mois': ventes_mensuelles.index.astype(str),
    'CA_Total': ventes_mensuelles.values,
    'CA_Moyen': (ventes_mensuelles / df.groupby("Mois").size()).round(2).values,
    'Nb_Transactions': df.groupby("Mois").size().values
})

print("\nAnalyse des ventes mensuelles :")
display(HTML(df_ventes_mensuelles.to_html(index=False)))

## Partie 4 : Visualisation des Données

In [ ]:
# Dossier de sauvegarde des figures
FIGURES_DIR = os.path.join("..", "reports", "figures")

In [ ]:
# Histogramme des ventes
fig = plt.figure(figsize=(10,5))
plt.hist(df["Chiffre_Affaires"], bins=20, edgecolor='black')
plt.title("Distribution du chiffre d'affaires par transaction")
plt.xlabel("Montant (€)")
plt.ylabel("Fréquence")
fig.savefig(os.path.join(FIGURES_DIR, "01_histogramme_ventes.png"), bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# Courbe temporelle
fig = plt.figure(figsize=(12,5))
ventes_mensuelles.plot(marker='o')
plt.title("Évolution mensuelle du chiffre d'affaires")
plt.ylabel("CA (€)")
plt.grid(True)
fig.savefig(os.path.join(FIGURES_DIR, "02_evolution_temporelle.png"), bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# Diagramme circulaire des catégories
ca_categorie = df.groupby("Catégorie")["Chiffre_Affaires"].sum()
fig = plt.figure(figsize=(8,8))
plt.pie(ca_categorie, labels=ca_categorie.index, autopct='%1.1f%%', startangle=90)
plt.title("Répartition du CA par catégorie")
fig.savefig(os.path.join(FIGURES_DIR, "03_categories.png"), bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# Boxplot des ventes
fig = plt.figure(figsize=(8,5))
sns.boxplot(y=df["Chiffre_Affaires"])
plt.title("Boxplot du chiffre d'affaires")
fig.savefig(os.path.join(FIGURES_DIR, "04_boxplot.png"), bbox_inches="tight", facecolor="white")
plt.show()

### Partie 5 : Analyse Avancée

In [ ]:
# 5.1 corrélation entre variables numériques

corr = df[["Quantité", "Prix_Unitaire", "Chiffre_Affaires"]].corr()
sns.heatmap(corr, annot=True)
plt.title("Matrice de corrélation")
plt.savefig(os.path.join(FIGURES_DIR, "06_correlation.png"), bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 5.2 les clients les plus fidèles
fidelite = df.groupby("Client_ID")["Date"].count().sort_values(ascending=False)
df_fidelite = df.groupby("Client_ID").agg({
    'Date': 'count',
    'Chiffre_Affaires': 'sum',
    'Quantité': 'sum'
}).rename(columns={'Date': 'Nb_Achats'})

df_fidelite = df_fidelite.sort_values('Nb_Achats', ascending=False).head(10).reset_index()

print("Top 10 des clients les plus fidèles :")

display(HTML(df_fidelite.to_html(index=False)))

In [ ]:
# 5.3 Segmenter les ventes par tranche de prix
df["Tranche_Prix"] = pd.cut(df["Prix_Unitaire"], bins=[0, 50, 200, 500, 2000], labels=["Bas", "Moyen", "Haut", "Premium"])
ventes_par_tranche = df.groupby("Tranche_Prix")["Chiffre_Affaires"].sum()

df_tranches = ventes_par_tranche.reset_index()
df_tranches.columns = ['Tranche_Prix', 'Chiffre_Affaires']

print("Ventes par tranche de prix :")
display(HTML(df_tranches.to_html(index=False)))

In [ ]:
# 5.4 analyse RFM simplifiée.
reference_date = df["Date"].max()
rfm = df.groupby("Client_ID").agg({
    "Date": lambda x: (reference_date - x.max()).days,
    "Client_ID": "count",
    "Chiffre_Affaires": "sum"
})
rfm.columns = ["Recence", "Frequence", "Montant"]
df_rfm = rfm.reset_index()

# Score RFM simple
df_rfm['RFM_Score'] = (
    (df_rfm['Recence'] < 30).astype(int) * 3 +
    (df_rfm['Recence'] < 90).astype(int) * 2 +
    (df_rfm['Frequence'] > 3).astype(int) * 3 +
    (df_rfm['Frequence'] > 1).astype(int) * 2 +
    (df_rfm['Montant'] > 500).astype(int) * 3 +
    (df_rfm['Montant'] > 200).astype(int) * 2
)

print("Segmentation RFM :")
display(HTML(df_rfm.to_html(index=False)))